In [0]:
%sql 
select * from metadata.pipeline_watermark

For gold layer taking incremntal data from silver layer for creating summary like

gold.fact_sales, 

gold.daily_sales_summary, 

gold.product_sales_summary, 

gold.customer_sales_summary,

gold.order_status_summary

In [0]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_date,
    sum as spark_sum, count, countDistinct,
    max as spark_max
)
from delta.tables import DeltaTable

watermark 

In [0]:
def get_last_watermark(table_name):
    result = spark.sql(f"""
        SELECT last_processed_timestamp
        FROM metadata.pipeline_watermark
        WHERE table_name = '{table_name}'
    """).collect()

    if len(result) == 0 or result[0]["last_processed_timestamp"] is None:
        return "1900-01-01 00:00:00"

    return result[0]["last_processed_timestamp"]


def update_watermark(table_name, new_watermark):
    if new_watermark is None:
        return

    watermark_df = spark.createDataFrame(
        [(table_name, new_watermark)],
        ["table_name", "last_processed_timestamp"]
    )

    target = DeltaTable.forName(spark, "metadata.pipeline_watermark")

    (
        target.alias("target")
        .merge(
            watermark_df.alias("source"),
            "target.table_name = source.table_name"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

merge fucntion for incremental merge and update 

In [0]:
def merge_to_gold(source_df, target_table, merge_keys):
    if not spark.catalog.tableExists(target_table):
        (
            source_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
    else:
        target = DeltaTable.forName(spark, target_table)

        merge_condition = " AND ".join(
            [f"target.{key} = source.{key}" for key in merge_keys]
        )

        (
            target.alias("target")
            .merge(source_df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

for gold.fct_sales

using following silver table : 

silver.orders,

silver.order_items,

silver.customers,

silver.products


combine all tables considering order_items and orders as out main principle table for sales

In [0]:
def build_fact_sales():
    table_name = "gold.fact_sales"
    last_wm = get_last_watermark(table_name)

    orders = spark.table("silver.orders")
    order_items = spark.table("silver.order_items")
    customers = spark.table("silver.customers")
    products = spark.table("silver.products")

    changed_orders = (
        orders
        .filter(col("silver_processed_at") > lit(last_wm))
        .select("order_id")
    )

    changed_items = (
        order_items
        .filter(col("silver_processed_at") > lit(last_wm))
        .select("order_id")
    )

    changed_order_ids = changed_orders.union(changed_items).distinct()

    if changed_order_ids.limit(1).count() == 0:
        print("No new records for gold.fact_sales")
        return

    fact_sales = (
        order_items.alias("oi")
        .join(changed_order_ids.alias("c"), "order_id", "inner")
        .join(orders.alias("o"), "order_id", "left")
        .join(customers.alias("cu"), col("o.customer_id") == col("cu.customer_id"), "left")
        .join(products.alias("p"), col("oi.product_id") == col("p.product_id"), "left")
        .select(
            col("oi.order_id"),
            col("oi.order_item_id"),
            col("o.customer_id"),
            col("cu.customer_unique_id"),
            col("cu.customer_city"),
            col("cu.customer_state"),
            col("cu.customer_segment"),
            col("oi.product_id"),
            col("p.product_category_name"),
            col("p.product_brand"),
            col("oi.seller_id"),
            col("o.order_status"),
            col("o.order_purchase_timestamp"),
            to_date(col("o.order_purchase_timestamp")).alias("order_date"),
            col("oi.shipping_limit_date"),
            col("oi.price"),
            col("oi.freight_value"),
            (col("oi.price") + col("oi.freight_value")).alias("gross_amount"),
            col("o.silver_processed_at").alias("order_silver_processed_at"),
            col("oi.silver_processed_at").alias("item_silver_processed_at")
        )
        .withColumn("gold_processed_at", current_timestamp())
    )

    merge_to_gold(
        fact_sales,
        "gold.fact_sales",
        ["order_id", "order_item_id"]
    )

    new_wm = fact_sales.agg(spark_max("gold_processed_at")).collect()[0][0]
    update_watermark(table_name, new_wm)

gold.daily_sales_suumary

In [0]:
def build_daily_sales_summary():
    table_name = "gold.daily_sales_summary"
    last_wm = get_last_watermark(table_name)

    fact = spark.table("gold.fact_sales")

    changed_dates = (
        fact
        .filter(col("gold_processed_at") > lit(last_wm))
        .select("order_date")
        .distinct()
    )

    if changed_dates.limit(1).count() == 0:
        print("No new records for gold.daily_sales_summary")
        return

    daily_summary = (
        fact
        .join(changed_dates, "order_date", "inner")
        .groupBy("order_date")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            count("*").alias("total_order_items"),
            spark_sum("price").alias("total_sales"),
            spark_sum("freight_value").alias("total_freight"),
            spark_sum("gross_amount").alias("total_gross_amount")
        )
        .withColumn("gold_processed_at", current_timestamp())
    )

    merge_to_gold(
        daily_summary,
        "gold.daily_sales_summary",
        ["order_date"]
    )

    new_wm = daily_summary.agg(spark_max("gold_processed_at")).collect()[0][0]
    update_watermark(table_name, new_wm)

gold.product_sales_summary

In [0]:
def build_product_sales_summary():
    table_name = "gold.product_sales_summary"
    last_wm = get_last_watermark(table_name)

    fact = spark.table("gold.fact_sales")

    changed_products = (
        fact
        .filter(col("gold_processed_at") > lit(last_wm))
        .select("product_id")
        .distinct()
    )

    if changed_products.limit(1).count() == 0:
        print("No new records for gold.product_sales_summary")
        return

    product_summary = (
        fact
        .join(changed_products, "product_id", "inner")
        .groupBy(
            "product_id",
            "product_category_name",
            "product_brand"
        )
        .agg(
            countDistinct("order_id").alias("total_orders"),
            count("*").alias("total_items_sold"),
            spark_sum("price").alias("total_sales"),
            spark_sum("freight_value").alias("total_freight"),
            spark_sum("gross_amount").alias("total_gross_amount")
        )
        .withColumn("gold_processed_at", current_timestamp())
    )

    merge_to_gold(
        product_summary,
        "gold.product_sales_summary",
        ["product_id"]
    )

    new_wm = product_summary.agg(spark_max("gold_processed_at")).collect()[0][0]
    update_watermark(table_name, new_wm)

golde.customer_sales_summary

In [0]:
def build_customer_sales_summary():
    table_name = "gold.customer_sales_summary"
    last_wm = get_last_watermark(table_name)

    fact = spark.table("gold.fact_sales")

    changed_customers = (
        fact
        .filter(col("gold_processed_at") > lit(last_wm))
        .select("customer_id")
        .distinct()
    )

    if changed_customers.limit(1).count() == 0:
        print("No new records for gold.customer_sales_summary")
        return

    customer_summary = (
        fact
        .join(changed_customers, "customer_id", "inner")
        .groupBy(
            "customer_id",
            "customer_unique_id",
            "customer_city",
            "customer_state",
            "customer_segment"
        )
        .agg(
            countDistinct("order_id").alias("total_orders"),
            count("*").alias("total_items"),
            spark_sum("price").alias("total_sales"),
            spark_sum("freight_value").alias("total_freight"),
            spark_sum("gross_amount").alias("total_gross_amount")
        )
        .withColumn("gold_processed_at", current_timestamp())
    )

    merge_to_gold(
        customer_summary,
        "gold.customer_sales_summary",
        ["customer_id"]
    )

    new_wm = customer_summary.agg(spark_max("gold_processed_at")).collect()[0][0]
    update_watermark(table_name, new_wm)

gold.order_status_summary

In [0]:
def build_order_status_summary():
    table_name = "gold.order_status_summary"
    last_wm = get_last_watermark(table_name)

    fact = spark.table("gold.fact_sales")

    changed_status = (
        fact
        .filter(col("gold_processed_at") > lit(last_wm))
        .select("order_status")
        .distinct()
    )

    if changed_status.limit(1).count() == 0:
        print("No new records for gold.order_status_summary")
        return

    status_summary = (
        fact
        .join(changed_status, "order_status", "inner")
        .groupBy("order_status")
        .agg(
            countDistinct("order_id").alias("total_orders"),
            count("*").alias("total_order_items"),
            spark_sum("price").alias("total_sales"),
            spark_sum("gross_amount").alias("total_gross_amount")
        )
        .withColumn("gold_processed_at", current_timestamp())
    )

    merge_to_gold(
        status_summary,
        "gold.order_status_summary",
        ["order_status"]
    )

    new_wm = status_summary.agg(spark_max("gold_processed_at")).collect()[0][0]
    update_watermark(table_name, new_wm)

run gold layer create and refresh

In [0]:


build_fact_sales()
build_daily_sales_summary()
build_product_sales_summary()
build_customer_sales_summary()
build_order_status_summary()

In [0]:
%sql
select * from gold.daily_sales_summary